In [1]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Qwen/Qwen2.5-0.5B-Instruct",
    local_dir="D:/huggingface_models/Qwen2.5-0.5B"
)

d:\Agentic-AI-Langraph\langgraph-tutorial\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 303.26it/s]


'D:\\huggingface_models\\Qwen2.5-0.5B'

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "D:/huggingface_models/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

print("Model loaded successfully!")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 334.27it/s]

Model loaded successfully!


In [3]:
from langgraph.graph import StateGraph,END,START
from typing import TypedDict


In [4]:
# create a state 

class LLMState(TypedDict):
    question:str
    answer:str
    

In [5]:
def llm_qa(state: LLMState) -> LLMState:
    question = state['question']

    messages = [
        {"role": "system", "content": "You are a helpful assistant. Give clear and accurate answers."},
        {"role": "user", "content": question}
    ]

    # return_dict=True returns a BatchEncoding; extract its input_ids tensor.
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    # Put inputs on the same device as the model's input embeddings.
    input_device = model.get_input_embeddings().weight.device
    input_ids = encoded["input_ids"].to(input_device)
    attention_mask = encoded.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(input_device)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # Remove the prompt tokens and decode only the generated answer.
    generated_tokens = outputs[0, input_ids.shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return {"answer": answer}


In [7]:
# create a graph
graph = StateGraph(LLMState)

#  add nodes
graph.add_node('llm_qa',llm_qa)

# add edge
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

# compile
workflow = graph.compile()

In [8]:
# execute

initial_state = {'question': 'How far is the Moon from the Earth?'}

final_state = workflow.invoke(initial_state)
print(final_state['answer'])


The distance between the Earth and the Moon is approximately 384,400 kilometers (238,855 miles). This measurement is based on the average distance between the two celestial bodies at their closest approach during full moon phases.
